# Breast Cancer Vital Status Classification

An academic team project comparing recorded Alive/Dead labels with a majority baseline, Logistic Regression and Random Forest. Dead is the positive class. Follow-up duration is excluded from the predictors.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd() if (Path.cwd() / "breast_cancer").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from breast_cancer.data import FEATURES, NUMERIC, TARGET, load_data, grouped_splits
from breast_cancer.experiment import run_experiment

OUTPUT = ROOT / "results/reproduced"
OUTPUT.mkdir(parents=True, exist_ok=True)

## Input and outcome

The bundled CSV contains 4,024 records. Exact duplicate removal leaves 4,023. Status describes the recorded vital status of people already diagnosed with breast cancer; it does not distinguish cancer from no cancer. Survival Months is follow-up information and is excluded from the 14 predictors.

In [ ]:
input_path = Path(os.environ.get("BREAST_DATA", str(ROOT / "data/Breast_Cancer.csv")))
frame, quality = load_data(input_path)
display(pd.Series({k: quality[k] for k in ["source_rows", "exact_duplicates_removed", "retained_rows", "unique_predictor_profiles", "repeated_predictor_profiles"]}, name="Data profile"))
display(frame[TARGET].map({0: "Alive", 1: "Dead"}).value_counts().rename("Recorded status"))

## Development data

A held-out test fold is reserved before model fitting. Identical predictor profiles remain together. Descriptive summaries below use only the development partition; missing-value statistics and transformations will be fitted within its training folds.

In [ ]:
development_ids, test_ids = grouped_splits(frame)[0]
development = frame.iloc[development_ids]
print(f"Development: {len(development)} rows; held-out test: {len(test_ids)} rows")
display(development[NUMERIC].describe().T.round(2))

In [ ]:
grade_status = pd.crosstab(development["Grade"], development[TARGET], normalize="index").rename(columns={0: "Alive", 1: "Dead"})
axis = grade_status.plot.bar(stacked=True, figsize=(7, 4), color=["#a6bddb", "#2166ac"])
axis.set(ylabel="Proportion within grade", title="Recorded status by grade in development data")
axis.figure.tight_layout()
axis.figure.savefig(OUTPUT / "grade_status.png", dpi=150)
plt.close(axis.figure)
display(Image(filename=str(OUTPUT / "grade_status.png")))

## Training and model selection

Five stratified group folds within development compare predefined class-weight variants. Each training fold fits its own imputer, scaler and one-hot vocabulary. No oversampling is performed. Highest mean CV average precision selects the model before the held-out test is scored. Classification uses a fixed threshold of 0.5.

In [ ]:
manifest = run_experiment(frame, quality, OUTPUT)

In [ ]:
cv_table = pd.DataFrame({name: {"CV mean AP": value["mean_average_precision"], "CV AP std": value["std_average_precision"]} for name, value in manifest["cv_results"].items()}).T
display(cv_table.round(4))
print("Selected by development CV:", manifest["selected_model"])

In [ ]:
test_table = pd.DataFrame(manifest["test_metrics"]).T
display(test_table[["accuracy", "balanced_accuracy", "dead_recall", "dead_precision", "average_precision", "roc_auc"]].astype(float).round(4))

In [ ]:
display(Image(filename=str(OUTPUT / "model_comparison.png")))
display(Image(filename=str(OUTPUT / "selected_confusion.png")))

## Interpretation

Average precision and ROC AUC assess ranking, while the confusion matrix depends on the fixed threshold. A high overall accuracy can coexist with low Dead recall. Class weighting changes that tradeoff, and the reported scores are not calibrated individual risks.

This is a retrospective vital-status classification benchmark with varying follow-up. It is not a fixed-horizon survival model. The source has no patient IDs, site metadata or diagnosis dates for external or temporal validation. See [the evaluation protocol](../docs/methodology.md) for details.

In [ ]:
print("Metrics:", OUTPUT / "metrics.json")
print("Saved preprocessing and classifier:", OUTPUT / "model.joblib")
print("Aggregate plots:", OUTPUT / "model_comparison.png")